# La foto de los negocios y la película de la economía, ¿cuentan la misma historia?

En este notebook juntamos **dos miradas** sobre Campeche que miden cosas muy distintas:

- **Lado micro (DENUE)** · *Abigail* — Cuenta **cuántos negocios hay** y a qué se dedican. Es una **foto**: un corte en el tiempo del directorio de establecimientos del INEGI.
- **Lado macro (ITAEE)** · *Aremy* — Mide **cómo se mueve la actividad económica** trimestre a trimestre. Es una **película**: una serie de tiempo.

La pregunta del proyecto:

> ¿La composición de las unidades económicas de Campeche se corresponde con el comportamiento de su actividad económica agregada?

Dicho sin tecnicismos: **¿tener muchísimos negocios de un tipo significa que ese tipo es el que realmente mueve la economía?** Spoiler: no, y aquí lo mostramos con datos.

---

## Qué vas a encontrar en este notebook

1. **Sección 1** — La consulta que une los dos mundos (un JOIN por `gran_division`) · *Aremy*
2. **Sección 2** — Una gráfica que los pone juntos, sin engañarnos con las escalas · *Abigail*
3. **Sección 3** — La conclusión, escrita para alguien que no sabe de bases de datos · *las dos*
4. **Sección 4** — Lo que NO podemos concluir y por qué · *las dos*

**Estado (20/ago/2026):** ambas capas cargadas y verificadas de punta a punta — BIE (4 series, 2022-Q1 a 2026-Q1) y DENUE (47,821 establecimientos). Si alguna capa aparece en 0 filas, reejecuta `python main.py` y vuelve a correr el notebook.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Para que funcione tanto desde la raíz del repo como desde integracion/.
RAIZ = Path.cwd()
if RAIZ.name == 'integracion':
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from core.db import consultar

# Para que los números se vean con 2 decimales y no con notación rara.
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Antes de empezar: ¿qué es `gran_division` y por qué es el puente?

Las dos capas vienen de fuentes distintas y **no tienen columnas en común**. Para unirlas usamos una clasificación que sí comparten: la **gran división** de la economía.

Toda actividad económica se agrupa en tres grandes bloques:

- **Primarias** — sacan recursos de la naturaleza (agricultura, ganadería, pesca).
- **Secundarias** — transforman cosas (manufactura, construcción, energía… y aquí vive la **extracción de petróleo**, clave para Campeche).
- **Terciarias** — servicios y comercio (tiendas, restaurantes, transporte, escuelas…).

El lado micro (DENUE) trae cada negocio con su `sector_id` (código SCIAN), y una tabla (`dim_sector_actividad`) traduce ese código a su `gran_division`. El lado macro (ITAEE) trae cada serie ya etiquetada con su `gran_division`. **Esa palabra compartida es lo que nos deja unir los dos mundos.**

> ⚠️ Ojo: el catálogo de sectores solo tiene `Primarias`, `Secundarias` y `Terciarias`. La serie `Total` del ITAEE **no tiene contraparte micro**, así que queda fuera de la comparación. Es correcto: la pregunta es de *composición*, no de totales.

## Sección 1 · La consulta que une los dos mundos *(Aremy)*

El plan es simple:

1. **Micro**: contar cuántos establecimientos hay por `gran_division`.
2. **Macro**: tomar el valor del ITAEE por `gran_division` y por trimestre.
3. **Pegar** ambas tablas por `gran_division`.

La consulta es un **JOIN** (unión). Primero la versión completa, que deja ver la serie de tiempo; luego una variante más simple que compara "aquí y ahora".

In [ ]:
# Versión completa: por cada gran_division, el conteo de negocios (fijo) y
# el valor del ITAEE en cada trimestre (que sí cambia con el tiempo).
SQL_JOIN = '''
WITH micro AS (
    -- Lado micro: cuántos negocios hay por gran división.
    SELECT d.gran_division,
           COUNT(*) AS n_establecimientos
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
),
macro AS (
    -- Lado macro: el valor del ITAEE por gran división y por periodo.
    SELECT bi.gran_division,
           o.anio,
           o.trimestre,
           o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'   -- '04' es Campeche
)
SELECT mi.gran_division,
       mi.n_establecimientos,
       ma.anio,
       ma.trimestre,
       ma.valor
FROM micro mi
JOIN macro ma ON mi.gran_division = ma.gran_division
ORDER BY mi.gran_division, ma.anio, ma.trimestre;
'''

df = consultar(SQL_JOIN)
print('Filas del JOIN:', len(df))
df.head()

In [ ]:
# Variante "aquí y ahora": compara el conteo de negocios contra el valor
# del ITAEE en el trimestre más reciente. El conteo no se repite por trimestre,
# así queda un cuadro limpio de 3 renglones (uno por gran división).
SQL_PUNTO = '''
WITH micro AS (
    SELECT d.gran_division, COUNT(*) AS n_establecimientos
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
),
ultimo AS (
    -- Encuentra el periodo más reciente de cada indicador.
    SELECT bi.gran_division, o.periodo
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
      AND o.periodo = (SELECT MAX(o2.periodo) FROM bie_observacion o2
                       WHERE o2.area_geografica = '04'
                         AND o2.indicador_id = o.indicador_id)
),
macro AS (
    SELECT bi.gran_division, o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    JOIN ultimo u ON u.gran_division = bi.gran_division
                 AND u.periodo = o.periodo
    WHERE o.area_geografica = '04'
)
SELECT m.gran_division,
       m.n_establecimientos,
       -- Qué porcentaje de TODOS los negocios representa cada gran división.
       ROUND(m.n_establecimientos * 100.0 / SUM(m.n_establecimientos) OVER (), 1) AS pct_establecimientos,
       ROUND(ma.valor, 2) AS itaee_ultimo_trimestre
FROM micro m
JOIN macro ma ON m.gran_division = ma.gran_division;
'''

df_punto = consultar(SQL_PUNTO)
print('Cuadro final (micro vs macro, último trimestre):')
df_punto

## Sección 2 · La gráfica *(Abigail)*

Aquí hay una trampa: **el conteo de negocios y el índice ITAEE no se pueden comparar directamente**.

- El conteo va de 0 a ~48 mil (son *unidades*: negocios).
- El ITAEE es un **índice base 2018 = 100** (un número *relativo*, no dinero ni cantidad).

Si los metemos en la misma gráfica con la misma escala, uno aplasta al otro y no se entiende nada. Nuestra solución es **separarlos en dos paneles**, cada uno con su propia escala y su propio mensaje:

- **Izquierda**: qué *porcentaje* de los negocios representa cada gran división (la foto).
- **Derecha**: cómo evoluciona el *índice* de cada gran división (la película).

Así leemos las dos cosas juntas **sin fingir que comparten escala**. La línea gris punteada en 100 marca el nivel base de 2018: por encima hay crecimiento respecto a 2018, por debajo hay caída.

In [ ]:
# Gráfica (Abigail): participación micro vs serie macro.
micro = consultar('''
    SELECT d.gran_division, COUNT(*) AS n
    FROM denue_establecimiento e
    JOIN dim_sector_actividad d ON e.sector_id = d.sector_id
    GROUP BY d.gran_division
''')

macro = consultar('''
    SELECT bi.gran_division, o.anio, o.trimestre, o.valor
    FROM bie_observacion o
    JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04'
''')
macro['periodo'] = macro['anio'].astype(str) + '-Q' + macro['trimestre'].astype(str)

orden = ['Primarias', 'Secundarias', 'Terciarias']
colores = ['#2ca02c', '#d62728', '#1f77b4']

fig, (ax_izq, ax_der) = plt.subplots(1, 2, figsize=(12, 4.5))

if micro.empty:
    ax_izq.text(0.5, 0.5, 'Sin datos micro (DENUE no cargado).\nReejecutar `python main.py`.',
                ha='center', va='center', transform=ax_izq.transAxes)
else:
    micro_pct = (micro.set_index('gran_division').loc[orden]['n'] / micro['n'].sum() * 100)
    barras = ax_izq.bar(micro_pct.index, micro_pct, color=colores)
    ax_izq.bar_label(barras, fmt='%.1f%%')
    ax_izq.set_ylabel('Participación en establecimientos (%)')
    ax_izq.set_title('Composición micro (DENUE): de cada 100 negocios...')
    ax_izq.set_ylim(0, 100)

for gd, color in zip(orden, colores):
    s = macro[macro['gran_division'] == gd].sort_values(['anio', 'trimestre'])
    ax_der.plot(s['periodo'], s['valor'], marker='o', markersize=3, label=gd, color=color)
ax_der.axhline(100, color='gray', lw=0.8, ls='--', alpha=0.6)
ax_der.set_ylabel('ITAEE (base 2018 = 100)')
ax_der.set_title('Actividad macro (BIE) — Campeche')
ax_der.legend(fontsize=8)
ax_der.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Números clave, reproducibles, para apoyar la conclusión.
if not micro.empty:
    total = micro['n'].sum()
    print(f'Establecimientos totales: {total:,}')
    print(micro.assign(pct=round(micro['n'] / total * 100, 1)).to_string(index=False))
else:
    print('Micro vacío (DENUE no cargado aún).')

print()
print('ITAEE en el último trimestre disponible (2026-Q1):')
print(consultar('''
    SELECT bi.gran_division, ROUND(o.valor, 2) AS valor
    FROM bie_observacion o JOIN bie_indicador bi ON o.indicador_id = bi.indicador_id
    WHERE o.area_geografica = '04' AND o.anio = 2026 AND o.trimestre = 1
''').to_string(index=False))

## Sección 3 · Conclusión *(las dos)*

No se corresponden. La composición de las unidades económicas de Campeche está dominada por las actividades terciarias, mientras que el comportamiento de su actividad agregada lo dictan otras. **Esa diferencia es el hallazgo, no un error de los datos.**

Del lado micro, los 47,821 establecimientos forman una economía de negocios pequeños: el **87% son terciarios** (41,706), encabezados por el comercio al por menor (18,916), otros servicios (7,213) y alimentos y hospedaje (6,756). Las primarias apenas representan **2.7%** del censo y las secundarias **10.1%**.

Del lado macro, el ITAEE (base 2018=100) cuenta otra historia. Las terciarias se sostienen como un piso estable alrededor del nivel base: los miles de comercios sostienen actividad, pero no la impulsan. Las secundarias —que son solo el 10% de los establecimientos— cayeron cerca de 20% entre 2022 y 2026 y se ubican más de 30 puntos por debajo de su base: un movimiento de ese tamaño con tan pocas unidades solo ocurre en sectores concentrados con gran peso en el producto. Las primarias, pocas pero volátiles, oscilan con la estacionalidad agropecuaria.

En una frase: **en Campeche los negocios que se cuentan por miles no son los que mueven el producto.** Medir la economía por el número de establecimientos diría que Campeche es una economía de servicios; medirla por su actividad agregada muestra que la dinámica la marcan unos cuantos sectores con pocas unidades y mucho peso.

## Sección 4 · Limitaciones *(las dos)*

- El DENUE cuenta **establecimientos**, no empleo ni producción. Una cadena con 40 sucursales aparece 40 veces, y los negocios sin registro pueden no estar en el directorio.
- El ITAEE es un **índice** (base 2018=100), no un monto: no se puede sumar entre divisiones ni afirmar cuánto "pesa" cada sector en el producto; solo se compara la trayectoria.
- **Fotografía vs. serie**: el DENUE es un corte en el tiempo; el ITAEE, una serie. Comparar su participación supone que la estructura actual de negocios es representativa de todo el periodo, y hay desfase: el censo es de 2026 y la serie arranca en 2022.
- **Estacionalidad de las primarias**: un solo trimestre como "nivel" sería engañoso; por eso la conclusión usa trayectorias, no niveles puntuales.
- No se puede concluir causalidad ni que "los comercios producen poco": para eso haría falta valor agregado por tamaño o estrato.
- El conteo incluye todos los tamaños (de micro a grande); sin el estrato como llave no sabemos cuánto empleo absorbe cada sector.
- El `Total` del ITAEE queda fuera del JOIN porque el catálogo de sectores no tiene una categoría "Total".
- La comparación depende de que ambas fuentes compartan el clasificador SCIAN 2018; un cambio de clasificador en cualquiera de las dos rompería el puente en silencio.